# Kernel SVM with a quantum feature map

Build a Gram matrix entry `K(s1, s2) = |⟨s1|s2⟩|²` via fidelity. Same trace runs `numpy.vdot` on cpu and SWAP-test on cpu+sim.

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the planner picks the route (cpu / cpu+sim / cpu+gpu) at preflight time.

## Background

**Problem:** kernel SVM. **Classical:** RBF kernel. **Quantum:** angle-encode features into states, fidelity between encoded states is the kernel matrix entry.

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
from uniqx.ops import fidelity

psi1 = np.array([0.6, 0.8])
psi2 = np.array([0.8, 0.6])

@trace
def prog(state1, state2):
    return fidelity(state1, state2)

s1 = np.concatenate([psi1, np.zeros_like(psi1)])
s2 = np.concatenate([psi2, np.zeros_like(psi2)])
module = prog(s1.tolist(), s2.tolist())


## Preflight — see what routes the planner found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the planner's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the planner picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")
